# Distance From Host

## Objective

Build one row per 2026 World Cup team with the distance between the team's country and the United States. I use the same OpenStreetMap geocoding approach from the distance from home notebook through `tidygeocoder::geocode(method = "osm")`.

The 2026 team list comes from FIFA's qualified teams page: https://www.fifa.com/en/tournaments/mens/worldcup/canadamexicousa2026/articles/world-cup-2026-who-has-qualified

## Output

- `1.DataCleaning-R/Data/RDS/WC2026DistanceFromHost.rds`


In [1]:
library(tidyverse)
library(tidygeocoder)
library(here)
library(worldcup)

Warning message:
"package 'ggplot2' was built under R version 4.4.3"
Warning message:
"package 'purrr' was built under R version 4.4.3"
-- Attaching core tidyverse packages ------------------------ tidyverse 2.0.0 --
v dplyr     1.1.4     v readr     2.1.5
v forcats   1.0.0     v stringr   1.6.0
v ggplot2   4.0.1     v tibble    3.2.1
v lubridate 1.9.4     v tidyr     1.3.1
v purrr     1.2.1     
-- Conflicts ------------------------------------------ tidyverse_conflicts() --
x dplyr::filter() masks stats::filter()
x dplyr::lag()    masks stats::lag()
i Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
here() starts at /Users/eialnisman/Desktop/WC2026Forecast



## Teams

The 2026 teams are entered manually because that tournament is not in the current `worldcup` package data.

In [2]:
teams_2026 <- tribble(
    ~team_name, ~team_code,
    "Canada", "CAN",
    "Mexico", "MEX",
    "United States", "USA",
    "Australia", "AUS",
    "Iraq", "IRQ",
    "Iran", "IRN",
    "Japan", "JPN",
    "Jordan", "JOR",
    "South Korea", "KOR",
    "Qatar", "QAT",
    "Saudi Arabia", "SAU",
    "Uzbekistan", "UZB",
    "Algeria", "DZA",
    "Cabo Verde", "CPV",
    "Congo DR", "COD",
    "Ivory Coast", "CIV",
    "Egypt", "EGY",
    "Ghana", "GHA",
    "Morocco", "MAR",
    "Senegal", "SEN",
    "South Africa", "ZAF",
    "Tunisia", "TUN",
    "Curacao", "CUW",
    "Haiti", "HTI",
    "Panama", "PAN",
    "Argentina", "ARG",
    "Brazil", "BRA",
    "Colombia", "COL",
    "Ecuador", "ECU",
    "Paraguay", "PRY",
    "Uruguay", "URY",
    "New Zealand", "NZL",
    "Austria", "AUT",
    "Belgium", "BEL",
    "Bosnia and Herzegovina", "BIH",
    "Croatia", "HRV",
    "Czechia", "CZE",
    "England", "ENG",
    "France", "FRA",
    "Germany", "DEU",
    "Netherlands", "NLD",
    "Norway", "NOR",
    "Portugal", "PRT",
    "Scotland", "SCO",
    "Spain", "ESP",
    "Sweden", "SWE",
    "Switzerland", "CHE",
    "Turkey", "TUR"
) %>%
    mutate(
        tournament_id = "WC-2026",
        package_team_name = case_when(
            team_name == "Czechia" ~ "Czech Republic",
            TRUE ~ team_name
        )
    ) %>%
    left_join(
        worldcup::teams %>% select(package_team_name = team_name, team_id),
        by = "package_team_name"
    ) %>%
    select(tournament_id, team_name, team_id, team_code)

teams_2026 %>% count(tournament_id)

tournament_id,n
<chr>,<int>
WC-2026,48


## Geocode Countries

A few football names need clearer country names for geocoding.

In [3]:
geocode_name <- function(country) {
    case_when(
        country == "Congo DR" ~ "Democratic Republic of the Congo",
        country == "Curacao" ~ "Curacao",
        country == "England" ~ "England, United Kingdom",
        country == "Iran" ~ "Iran",
        country == "Ivory Coast" ~ "Cote d'Ivoire",
        country == "Scotland" ~ "Scotland, United Kingdom",
        country == "South Korea" ~ "South Korea",
        country == "United States" ~ "United States of America",
        TRUE ~ country
    )
}

country_locations <- teams_2026 %>%
    distinct(team_name) %>%
    mutate(geocode_query = geocode_name(team_name)) %>%
    geocode(geocode_query, method = "osm", lat = country_lat, long = country_long)

host_location <- tribble(
    ~tournament_id, ~host_country, ~host_geocode_query,
    "WC-2026", "United States", "United States of America"
) %>%
    geocode(host_geocode_query, method = "osm", lat = host_lat, long = host_long)

Passing 48 addresses to the Nominatim single address geocoder

Query completed in: 48.3 seconds

Passing 1 address to the Nominatim single address geocoder

Query completed in: 1 seconds



## Calculate Distance

Distance is great-circle distance in kilometers. The United States is used as the host country. The United States, Canada, and Mexico get 0 because they are co-hosts.

In [4]:
haversine_km <- function(lat1, lon1, lat2, lon2) {
    earth_radius_km <- 6371
    to_radians <- function(degrees) degrees * pi / 180

    dlat <- to_radians(lat2 - lat1)
    dlon <- to_radians(lon2 - lon1)
    lat1 <- to_radians(lat1)
    lat2 <- to_radians(lat2)

    a <- sin(dlat / 2)^2 + cos(lat1) * cos(lat2) * sin(dlon / 2)^2
    2 * earth_radius_km * atan2(sqrt(a), sqrt(1 - a))
}

home_countries <- tribble(
    ~tournament_id, ~home_country,
    "WC-2026", "United States",
    "WC-2026", "Mexico",
    "WC-2026", "Canada"
)

distance_from_host <- teams_2026 %>%
    left_join(country_locations, by = "team_name") %>%
    left_join(host_location, by = "tournament_id") %>%
    left_join(
        home_countries %>% mutate(is_home_country = TRUE),
        by = c("tournament_id", "team_name" = "home_country")
    ) %>%
    mutate(
        is_home_country = replace_na(is_home_country, FALSE),
        distance_from_host_km = if_else(
            is_home_country,
            0,
            haversine_km(country_lat, country_long, host_lat, host_long)
        )
    ) %>%
    select(
        tournament_id,
        team_name,
        team_id,
        team_code,
        host_country,
        is_home_country,
        country_lat,
        country_long,
        host_lat,
        host_long,
        distance_from_host_km
    ) %>%
    arrange(team_name)

distance_from_host %>%
    summarize(
        teams = n(),
        missing_country_coordinates = sum(is.na(country_lat) | is.na(country_long)),
        missing_host_coordinates = sum(is.na(host_lat) | is.na(host_long)),
        home_country_rows = sum(is_home_country),
        .groups = "drop"
    )

teams,missing_country_coordinates,missing_host_coordinates,home_country_rows
<int>,<int>,<int>,<int>
48,0,0,3


Inspect

In [5]:
distance_from_host

tournament_id,team_name,team_id,team_code,host_country,is_home_country,country_lat,country_long,host_lat,host_long,distance_from_host_km
<chr>,<chr>,<chr>,<chr>,<chr>,<lgl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
WC-2026,Algeria,T-01,DZA,United States,FALSE,36.772933,3.058845,39.78373,-100.4459,8467.886
WC-2026,Argentina,T-03,ARG,United States,FALSE,-34.996496,-64.967282,39.78373,-100.4459,9076.351
WC-2026,Australia,T-04,AUS,United States,FALSE,-24.776109,134.755000,39.78373,-100.4459,14653.989
WC-2026,Austria,T-05,AUT,United States,FALSE,47.593970,14.124560,39.78373,-100.4459,8351.637
WC-2026,Belgium,T-06,BEL,United States,FALSE,50.640281,4.666715,39.78373,-100.4459,7608.706
WC-2026,Bosnia and Herzegovina,T-08,BIH,United States,FALSE,44.305348,17.596147,39.78373,-100.4459,8799.915
WC-2026,Brazil,T-09,BRA,United States,FALSE,-10.333333,-53.200000,39.78373,-100.4459,7396.652
WC-2026,Cabo Verde,NA,CPV,United States,FALSE,16.000055,-24.008395,39.78373,-100.4459,7732.128
WC-2026,Canada,T-12,CAN,United States,TRUE,61.066692,-107.991707,39.78373,-100.4459,0.000


## Save

In [6]:
saveRDS(distance_from_host, here("1.DataCleaning-R", "Data", "RDS", "WC2026DistanceFromHost.rds"))